# Amortized Inference for a NLME Model

In [ ]:
# load necessary packages
import numpy as np
import pandas as pd
import itertools

# for plots
import matplotlib.pyplot as plt

from inference.inference_functions import run_population_optimization, create_boundaries_from_prior
from inference.nlme_objective import ObjectiveFunctionNLME
from inference.helper_functions import transform_pesto_results, compute_error_estimate

In [ ]:
# specify which model to use
model_name = ['fröhlich-small', 'fröhlich-large', 'fröhlich-sde', 'pharmacokinetic_model'][0]

## Load model

In [ ]:
if model_name == 'fröhlich-small':
    from models.froehlich_model_small import FroehlichModelSmall
    model = FroehlichModelSmall(load_best=True)

    use_presimulation = False
elif model_name == 'fröhlich-large':
    from models.froehlich_model_large import FroehlichModelLarge
    model = FroehlichModelLarge(load_best=True)

    use_presimulation = True
    presimulation_path = 'data/presimulations_froehlich_large'

elif model_name == 'fröhlich-sde':
    from models.froehlich_model_sde import FroehlichModelSDE
    model = FroehlichModelSDE(load_best=True)

    use_presimulation = True
    presimulation_path = 'data/presimulations_froehlich_sde'

elif model_name == 'pharmacokinetic_model':
    from models.pharmacokinetic_model import PharmacokineticModel
    model = PharmacokineticModel(load_best=True)

    use_presimulation = True
    presimulation_path = 'data/presimulations_pharma'
else:
    raise NotImplementedError('model not implemented')

path_store_network = 'networks/' + model.network_name

# assemble simulator and prior
simulator =  model.build_simulator()
trainer = model.build_trainer(path_store_network)

In [ ]:
model.print_and_plot_example()

## Estimating Population Parameters

Now we want to use the amortizer to generate samples such that we can minimize the negative log-likelihood of the data given the population parameters of the mixed effect model:
$$
    \beta^*,\Psi^* \approx 
    \underset{\beta,\Psi}{\arg\min} -\sum_i \log\left( \frac1M \sum_j^M \frac{p(\phi_j \mid \beta,\Psi)}{p(\phi_j)} \right).
$$

Remark: the objective value is not the likelihood value since the sum over $\log p(y_i)$ is missing.
One part of the objective function increases logarithmically with the sample size $M$.
On the other hand, the approximation gets better (therby decreasing the value?!)

$\beta$ is called ```theta_population``` in the code.


$\log \phi$ cell specific parameters, sampled from $\mathcal{N}(\beta,\Psi)$
$$
    p( \phi \mid \beta,\Psi) = (2\pi)^{-k/2}\vert \Psi\vert^{-1/2} \prod_{l=1}^k \phi_l^{-1} \exp \left(-\frac12 (\log \phi-\beta)^T \Psi^{-1}  (\log \phi-\beta) \right)
$$

Assumptions to start with: $\Psi$ is a diagonal matrix, need better parameterization for other types

$$
    \beta^*,\Psi^* \approx
    \underset{\beta,\Psi}{\arg\min} -\sum_i \left(\log \frac1M \sum_j^M \frac{p( \phi_j \mid \beta,\Psi)}{p( \phi_j)} \right) \\
     =  \underset{\beta,\Psi}{\arg\min} -\sum_i \left(\log\left(\vert \Psi\vert^{-1/2} \right) -\log M -
    \log\left(\vert \Sigma\vert^{-1/2}\right) +\log \sum_j^M \exp \left(-\frac12 (\log\phi_j-\beta)^T \Psi^{-1}  (\log\phi_j-\beta) + \frac12 (\log\phi_j-\mu)^T \Sigma^{-1}  (\log\phi_j-\mu) \right)\right)
$$

if the prior is $p( \phi) = (2\pi)^{-k/2}\vert \Sigma\vert^{-1/2} \prod_{l=1}^k \phi_l^{-1}\exp \left(-\frac12 (\log \phi-\mu)^T \Sigma^{-1}  (\log\phi-\mu) \right)$.


For purpose of optimization we also parametrize $\Psi$ by a log-transformation since diagonal entries must be positive.

# Define Objective Function

In [ ]:
cov_type = ['diag', 'cholesky'][0]
obj_fun_amortized = ObjectiveFunctionNLME(model_name=model.name,
                                          param_samples=np.empty((1,1,1)),
                                          prior_mean=model.prior_mean,
                                          prior_std=model.prior_std,
                                          covariance_format=cov_type,
                                          # np.exp(-3.41) * 1.5  # 1.5 times the median of the posterior standard deviations
                                          penalize_correlations=None,
                                          huber_loss_delta=None)

In [ ]:
pop_param_names = ['pop-' + name for name in model.param_names]
var_param_names = ['var-' + name for name in model.param_names]
full_param_names = pop_param_names + var_param_names

if cov_type == 'cholesky' and len(full_param_names) == model.n_params*2:
    # add  correlation names to full parameter names
    # make all possible combinations of parameter names
    combinations = list(itertools.combinations(model.param_names, 2))
    # create upper triangular matrix of parameter names
    psi_inverse_upper = np.chararray((model.n_params, model.n_params), itemsize=100, unicode=True)
    psi_inverse_upper[np.diag_indices(model.n_params)] = "1"
    psi_inverse_upper[np.triu_indices(model.n_params, k=1)] = [f"corr_{x}_{y}" for x, y in combinations]
    # extract lower triangular matrix of parameter names, so that they are ordered as in the covariance matrix
    corr_names = list(psi_inverse_upper.T[np.tril_indices(model.n_params, k=-1)])
    # add correlation names to full parameter names
    full_param_names = full_param_names + corr_names
    print(full_param_names)

Correlation

In [ ]:
obs_data, true_pop_parameters = model.load_data(n_data=50, load_eGFP=False, load_d2eGFP=False)

param_samples = model.draw_posterior_samples(data=obs_data, n_samples=1000)
param_median = np.median(param_samples, axis=1)
# correlations without sigma
median_df = pd.DataFrame(param_median[:, :-1], columns=model.param_names[:-1])
corr_df = median_df.corr()

In [ ]:
if cov_type == 'cholesky':
    # Find pairs with correlation above the threshold
    threshold = 0.2
    abs_corr_matrix = corr_df.abs()
    high_corr_pairs = abs_corr_matrix[abs_corr_matrix > threshold].unstack()
    #high_corr_pairs = high_corr_pairs[high_corr_pairs['level_0'] != high_corr_pairs['level_1']]
    high_corr_pairs = high_corr_pairs[high_corr_pairs < 1].sort_values(ascending=False)

    # find indices corresponding to pairs with high correlations
    high_corr_pairs_index = []
    high_corr_pairs_names = []
    for x,y in high_corr_pairs.index:
        name = f'corr_{x}_{y}'
        high_corr_pairs_index += [p_i for p_i, p_name in enumerate(full_param_names) if name == p_name]
        high_corr_pairs_names += [p_name for p_i, p_name in enumerate(full_param_names) if name == p_name]
    print(high_corr_pairs_names, len(high_corr_pairs_index))

# Optimization (Single Experiment)

In [ ]:
# create boundaries
lower_bound, upper_bound = create_boundaries_from_prior(
        prior_mean=model.prior_mean,
        prior_std=model.prior_std,
        boundary_width_from_prior=2.58,  # 99% of the prior mass is within 2.58 standard deviations
        covariance_format=cov_type
)

# sigma variance is always fixed
index_sigma = [ix for ix, x in enumerate(full_param_names) if 'var-$\\sigma$' == x]
fixed_indices = np.array(index_sigma)  # variance of sigma fixed
fixed_vals = np.array([upper_bound[index_sigma[0]]])

only_allow_high_corr = True
if cov_type == 'cholesky':
    if only_allow_high_corr:
        # fix all correlations to 0 but high correlated ones
        index_all_corr = [ix for ix, x in enumerate(full_param_names) if 'corr' in x]
        # remove high_corr_pairs_index from index_all_corr
        index_all_corr = [ix for ix in index_all_corr if ix not in high_corr_pairs_index]
        fixed_indices = np.append(fixed_indices, index_all_corr)
        fixed_vals = np.append(fixed_vals, np.zeros(len(index_all_corr)))
    else:
        # fix all correlations with sigma to 0
        index_sigma_corr = [ix for ix, x in enumerate(full_param_names) if 'sigma' in x and 'corr' in x]
        fixed_indices = np.append(fixed_indices, index_sigma_corr)
        fixed_vals = np.append(fixed_vals, np.zeros(len(index_sigma_corr)))

## Test Scalability

Testing dependency on number of data points and number of samples from the posterior.

In [ ]:
def test_scalability(
    test_n_cells: list[int],
    n_samples_opt_list: list[int],
    n_multi_starts: int = 1,
    noise_on_start_param: int = 2,
    multi_experiments: bool = False,
    data_name: str = ''
):
    # prepare placeholders
    time_opt = np.zeros((len(test_n_cells), len(n_samples_opt_list), n_multi_starts))
    rel_error = np.zeros((len(test_n_cells), len(n_samples_opt_list), n_multi_starts))

    assert n_multi_starts % 8 == 0, 'n_multi_starts must be a multiple of 8 for this function'

    print(data_name)
    for i, n_cells in enumerate(test_n_cells):
        print('\ntesting', n_cells, 'cells')

        if not multi_experiments:
            # load data
            obs_data, true_pop_parameters = model.load_data(n_data=n_cells, load_eGFP=False, load_d2eGFP=False)
        else:
            return NotImplementedError

        for j, n_samples_opt in enumerate(n_samples_opt_list):
            print('--testing', n_samples_opt, 'samples')

            # start counter
            result_i_optimization = run_population_optimization(
    bf_amortizer=model.amortizer,
    data=obs_data,
    param_names=full_param_names,
    objective_function=obj_fun_amortized,
    sample_posterior=model.draw_posterior_samples,
    n_multi_starts=n_multi_starts,
    noise_on_start_param=noise_on_start_param,
    n_samples_opt=n_samples_opt,
    lb=lower_bound,
    ub=upper_bound,
    x_fixed_indices=fixed_indices,
    x_fixed_vals=fixed_vals,
    file_name=f'output/scalability/{model.name}_cells_{n_cells}_samples_{n_samples_opt}.hd5',
    verbose=False,
    trace_record=False,
    pesto_multi_processes=8,
    )

            # transform results
            results_i = result_i_optimization.optimize_result.as_dataframe()['x']
            results_i_transformed = transform_pesto_results(results_i, int(len(results_i[0])/2))
            
            # get scalability measurements
            time_opt[i, j, :] = result_i_optimization.optimize_result.as_dataframe()['time']
            np.save(f'output/scalability/synthetic_time_opt_{data_name}.npy', time_opt)

            # compute relative error of parameter estimated as minimum over multi_starts
            if true_pop_parameters is not None:
                rel_error[i, j, :] = compute_error_estimate(results_i_transformed,
                                                           true_pop_parameters,
                                                           small_model=True if data_name == '' else False)
                np.save(f'output/scalability/synthetic_rel_error_{data_name}.npy', rel_error)

    return time_opt, rel_error

In [ ]:
test_n_cells = [50, 100, 200, 500, 1000, 5000, 10000]
n_samples_opt_list = [10, 50, 100] # [2, 10, 25, 50, 75, 100]
multi_experiments = False

In [ ]:
%%time
data_name = ''  # fröhlich-small
if model_name == 'fröhlich-large' :
    data_name = '_large_model'
elif model_name == 'fröhlich-sde':
    data_name = '_sde_model'

time_opt, rel_error = test_scalability(
    n_multi_starts=56,
    noise_on_start_param=1,
    test_n_cells=test_n_cells,
    n_samples_opt_list=n_samples_opt_list,
    multi_experiments=multi_experiments,
    data_name=data_name
)

In [ ]:
if multi_experiments:
    time_opt = np.load('output/real_time_opt.npy')
    rel_error = np.load('output/real_rel_error.npy')
else:
    time_opt = np.load('output/synthetic_time_opt.npy')
    rel_error = np.load('output/synthetic_rel_error.npy')

In [ ]:
# read results from monolix
monolix_errors = np.zeros(len(test_n_cells))
for cell_idx, n_cells in enumerate(test_n_cells):
    if n_cells == 1000: continue
    results_monolix = pd.read_csv(f'output/results_monolix/estimated_parameters_synthetic_{n_cells}_cells.csv',
                                 index_col=0, header=0)
    true_sample_parameters = pd.read_csv(f'data/synthetic/sample_pop_parameters.csv',
                                      index_col=0, header=0).loc[f'{n_cells}'].values
    results_to_compare = results_monolix[results_monolix.columns[0]].values[[0,1,2,3,4,10,5,6,7,8,9]]
    results_to_compare[5] = np.log(results_to_compare[5])
    results_to_compare = np.concatenate((results_to_compare, [0]))[np.newaxis,:]
    monolix_errors[cell_idx] = np.min(compute_error_estimate(results_to_compare, true_sample_parameters))

In [ ]:
figure, axis = plt.subplots(nrows=1, ncols=2, sharex='col', sharey='row', figsize=(15,5))
    
for j, n_samples_opt in enumerate(n_samples_opt_list): 
    axis[0].plot(test_n_cells, rel_error[:, j], label=f'#posterior samples: {n_samples_opt}')
    axis[1].scatter(np.array(test_n_cells)*n_samples_opt, rel_error[:, j],
                       label=f'#posterior samples: {n_samples_opt}')
    
axis[0].plot(test_n_cells, monolix_errors, label=f'baseline')

axis[0].set_ylabel('$t\,[s]$')
axis[0].set_ylabel('Relative mean squared error')
axis[0].set_xscale('log')
axis[1].set_xscale('log')
axis[0].set_yscale('log')
axis[0].set_title('Error (compared to true population parameters)')
axis[1].set_title('Error (compared to true population parameters)')
axis[0].set_xlabel('#cells')
axis[0].set_xticks(ticks=test_n_cells, rotation=60)
axis[1].set_xlabel('#cells $\cdot$ #posterior samples')
axis[0].legend()
axis[1].legend()
#axis[1].set_ylim(0, 8)
plt.tight_layout()

if multi_experiments:
    pass
    #plt.savefig('plots/real_time_error_vs_#cells_#samples.png')
else:
    pass
    #plt.savefig('plots/synthetic_time_error_vs_#cells_#samples.png')
plt.show()

In [ ]:
time_monolix = pd.read_csv(f'output/results_monolix/average_optimal_timings.csv', index_col=1, header=0)['average_time']

In [ ]:
figure, axis = plt.subplots(nrows=1, ncols=2, sharey='row', figsize=(15,5))


axis[0].hlines(4.2, xmin=test_n_cells[0], xmax=test_n_cells[-1], color='grey', linestyle='--', label=f'average training time of BayesFlow')

for j, n_samples_opt in enumerate(n_samples_opt_list): 
    axis[0].plot(test_n_cells, time_opt[:, j]/60/60, label=f'#posterior samples: {n_samples_opt}')
    axis[1].scatter(np.array(test_n_cells)*n_samples_opt, time_opt[:, j]/60/60,
                       label=f'#posterior samples: {n_samples_opt}')
    
axis[0].plot(test_n_cells, time_monolix, label=f'baseline')

        
# joint settings
axis[0].set_ylabel('$t\,[h]$')
#axis[0].set_yscale('log')
#axis[1].set_yscale('log')
axis[0].set_xscale('log')
axis[1].set_xscale('log')
axis[0].set_title('Optimization Time For a New Data Set')
axis[1].set_title('Optimization Time For a New Data Set')
axis[0].legend()
axis[1].legend()

# other settings
axis[0].set_xlabel('#cells')
axis[0].set_xticks(ticks=test_n_cells, rotation=60)
axis[1].set_xlabel('#cells $\cdot$ #posterior samples')
#axis[1].set_ylim(1e-3, 1e6)
plt.tight_layout()
#plt.savefig('plots/synthetic_time_vs_#cells_#samples.png')
plt.show()